# 1. Instalaciones

In [41]:
%pip install langchain-text-splitters tiktoken
%pip install azure-storage-blob python-dotenv 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Importaciones

In [42]:
import os
import json
import hashlib
import re
import time
from datetime import datetime
from typing import Dict, Any, List
from azure.storage.blob import BlobServiceClient, ContentSettings
from dotenv import load_dotenv
import traceback
import unicodedata
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 3. Configuración

In [43]:
load_dotenv()

CONNECTION_STRING = os.getenv("BLOB_CONNECTION_STRING")
CONTAINER_NAME = os.getenv("BRONZE_CONTAINER_NAME")
EXTRACTED_PREFIX = os.getenv("BLOB_EXTRACTED", "servicio_policia/servicio_policia_extracted/")
PROCESSED_PREFIX = os.getenv("BLOB_PROCESSED", "servicio_policia/servicio_policia_processed/")


# Validación de variables críticas
required_vars = ["BLOB_CONNECTION_STRING"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Faltan variables de entorno: {', '.join(missing)}")

# Inicializar cliente Storage
blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

print(f"Prefijo extracted: {EXTRACTED_PREFIX}")
print(f"Prefijo processed: {PROCESSED_PREFIX}")

Prefijo extracted: servicio_policia/servicio_policia_extracted/
Prefijo processed: servicio_policia/servicio_policia_processed/


# 4. Funciones para cargar documentos

In [44]:
def list_extracted_documents() -> List[str]:
    """Lista todos los documentos JSON en el prefijo extraído"""
    blobs = container_client.list_blobs(name_starts_with=EXTRACTED_PREFIX)
    return [blob.name for blob in blobs if blob.name.lower().endswith('.json')]

def load_json_from_bronze(blob_name: str) -> Dict[str, Any]:
    """Carga un documento JSON desde el contenedor bronze"""
    blob_client = container_client.get_blob_client(blob_name)
    download_stream = blob_client.download_blob()
    content = download_stream.readall().decode('utf-8')
    return json.loads(content)

# 5. Funciones de limpieza

In [ ]:
def clean_text(text: str) -> str:
    """Limpia el texto eliminando caracteres extraños y normalizando espacios"""
    if not text:
        return ""
    
    # Normalizar caracteres Unicode
    text = unicodedata.normalize('NFKC', text)
    
    # Eliminar caracteres de control (excepto saltos de línea y tabulaciones)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    
    # Normalizar espacios: múltiples espacios a uno solo
    text = re.sub(r'\s+', ' ', text)
    
    # Eliminar espacios al inicio y final
    text = text.strip()
    
    # Normalizar saltos de línea: múltiples saltos a dos saltos
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text

# 6. Creación de chunks

In [ ]:
class TextChunker:
    """Chunks text documents into smaller pieces for indexing"""
    
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        
        # Create text splitter
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
    
    def _find_page_range(self, start_idx: int, end_idx: int, offset_map: List[Dict]) -> tuple:
        """Find the start and end pages for a given text range using the offset map"""
        if not offset_map:
            return 1, 1, None
            
        start_page = None
        end_page = None
        section = None
        
        # Find overlapping spans
        for span in offset_map:
            # Check for overlap between span and chunk range
            if span['end'] > start_idx and span['start'] < end_idx:
                if start_page is None:
                    start_page = span['page']
                end_page = span['page']
                
                # Get section from the first overlapping span
                if section is None and 'section' in span:
                    section = span['section']
        
        if start_page is None:
            return 1, 1, None
            
        return start_page, end_page, section

    def chunk_document(self, document: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Chunk a single document into smaller pieces with page metadata
        Returns list of chunks in the target format
        """
        # Extract content from the new format
        content = document.get("extraction_results", {}).get("content", "")
        content = clean_text(content)  # Clean the content
        
        offset_map = document.get("offset_map", [])
        basic_metadata = document.get("basic_metadata", {})
        document_analysis = document.get("document_analysis", {})
        
        if not content or not content.strip():
            print(f"Warning: Empty content for {basic_metadata.get('file_name', 'unknown')}")
            return []
        
        # Split text into chunks
        chunks = self.text_splitter.split_text(content)
        
        # Create chunk documents with metadata
        chunk_docs = []
        cursor = 0
        
        for i, chunk_text in enumerate(chunks):
            # Clean chunk text
            chunk_text = clean_text(chunk_text)
            
            # Find exact position of this chunk in original text
            start_index = content.find(chunk_text, cursor)
            
            if start_index == -1:
                # Fallback: use approximate pages
                page_start, page_end, section = 1, 1, None
            else:
                end_index = start_index + len(chunk_text)
                page_start, page_end, section = self._find_page_range(start_index, end_index, offset_map)
                cursor = start_index + 1
       
            
            # Prepare metadata
            file_name = basic_metadata.get("file_name", "")
            file_path = basic_metadata.get("file_path", "")
            file_extension = basic_metadata.get("file_extension", "")
            total_pages = document_analysis.get("total_pages", 1)
            
            # Clean file extension
            if file_extension.startswith('.'):
                file_type = file_extension[1:].lower()
            else:
                file_type = file_extension.lower()
            
            # Create chunk_id (base name without extension)
            base_name = os.path.splitext(file_name)[0]
            chunk_id = f"{base_name}_{i}"
            
            # Create unique ID using MD5 hash of chunk_id
            unique_id = hashlib.md5(chunk_id.encode()).hexdigest()
            
            # Create blob_url
            blob_url = f"file://{file_path}"
            
            # Create chunk in target format
            chunk_doc = {
                "id": unique_id,
                "chunk_id": chunk_id,
                "content": chunk_text,
                "filename": file_name,
                "filepath": file_path,
                "blob_url": blob_url,
                "pages": total_pages,
                "pages_total": total_pages,
                "page_start": page_start,
                "page_end": page_end,
                "section": section,
                "file_type": file_type,
                "chunk_index": i,
                "total_chunks": len(chunks),
                "processing_timestamp": datetime.utcnow().isoformat()
            }
            chunk_docs.append(chunk_doc)
        
        return chunk_docs

# 7. Procesamiento de un solo documento

In [47]:
def process_single_document(blob_name: str, chunker: TextChunker) -> Dict[str, Any]:
    """Procesa un documento individual desde extracted a processed"""
    try:
        print(f"Procesando documento: {blob_name}")
        
        # 1. Cargar desde bronze
        original_data = load_json_from_bronze(blob_name)
        
        # 2. Verificar estructura básica
        if not original_data.get("extraction_results"):
            raise ValueError("Documento no tiene resultados de extracción")
        
        # 3. Crear chunks usando TextChunker
        chunks = chunker.chunk_document(original_data)
        
        if not chunks:
            raise ValueError("No se pudieron crear chunks del documento")
        
        # 4. Crear el documento procesado (lista de chunks)
        original_filename = os.path.basename(blob_name)
        processed_filename = f"{PROCESSED_PREFIX}{original_filename}"
        
        return {
            "status": "success",
            "original_blob": blob_name,
            "processed_blob": processed_filename,
            "chunks": chunks,
            "chunks_count": len(chunks),
            "document_id": hashlib.md5(blob_name.encode()).hexdigest()[:12]
        }
        
    except Exception as e:
        print(f"Error procesando {blob_name}: {str(e)}")
        traceback.print_exc()
        return {
            "status": "error",
            "original_blob": blob_name,
            "error": str(e),
            "chunks": []
        }

# 8. Guardado en processed

In [48]:
def save_to_processed(processed_result: Dict[str, Any]) -> str:
    """Guarda el documento procesado (lista de chunks) en processed"""
    try:
        blob_name = processed_result["processed_blob"]
        chunks = processed_result["chunks"]
        
        # Convertir lista de chunks a JSON
        json_content = json.dumps(
            chunks, 
            indent=2, 
            ensure_ascii=False,
            default=lambda obj: obj.isoformat() if hasattr(obj, 'isoformat') else str(obj)
        )
        
        # Subir a processed
        blob_client = container_client.get_blob_client(blob_name)
        blob_client.upload_blob(
            json_content,
            overwrite=True,
            content_settings=ContentSettings(content_type='application/json')
        )
        
        print(f"  ✓ Guardado en processed: {blob_name}")
        print(f"  ✓ Chunks guardados: {len(chunks)}")
        return blob_name
        
    except Exception as e:
        print(f"  ✗ Error guardando en processed: {str(e)}")
        raise

# 9. Procesamiento por lotes

In [49]:
def process_documents_batch(max_documents: int = 500, delay_between: float = 1.0):
    """Procesa un lote de documentos desde extracted a processed"""
    
    # Inicializar chunker
    chunker = TextChunker(chunk_size=1000, chunk_overlap=200)
    
    # Listar documentos en bronze
    print("Buscando documentos en bronze...")
    documents_to_process = list_extracted_documents()
    
    if not documents_to_process:
        print("No se encontraron documentos para procesar")
        return [], []
    
    if max_documents:
        documents_to_process = documents_to_process[:max_documents]
    
    print(f"Procesando {len(documents_to_process)} documentos...")
    print("-" * 60)
    
    processed = []
    errors = []
    
    for i, blob_name in enumerate(documents_to_process, 1):
        try:
            print(f"[{i}/{len(documents_to_process)}] {blob_name}")
            
            # Procesar documento
            result = process_single_document(blob_name, chunker)
            
            if result["status"] == "success":
                # Guardar en processed
                saved_name = save_to_processed(result)
                
                processed.append({
                    "original": blob_name,
                    "processed": saved_name,
                    "document_id": result["document_id"],
                    "chunks": result["chunks_count"],
                    "success": True
                })
                
                print(f"  Documento ID: {result['document_id']}")
                print(f"  Chunks creados: {result['chunks_count']}")
                
            else:
                errors.append({
                    "original": blob_name,
                    "error": result["error"],
                    "success": False
                })
                print(f"  ✗ Error: {result['error'][:100]}...")
            
            # Esperar entre procesamientos
            if i < len(documents_to_process):
                time.sleep(delay_between)
                
            print()
            
        except Exception as e:
            error_info = {
                "original": blob_name,
                "error": str(e),
                "success": False
            }
            errors.append(error_info)
            print(f"  ✗ Error crítico: {str(e)}")
            print()
    
    return processed, errors

# 10. Ejecución

In [50]:
def main():
    """Función principal para ejecutar el procesamiento"""
    print("=" * 60)
    print("PROCESAMIENTO EXTRACTED → PROCESSED")
    print("=" * 60)
    print(f"Processed container: {CONTAINER_NAME}")
    print(f"Documentos fuente: {EXTRACTED_PREFIX}")
    print(f"Documentos destino: {PROCESSED_PREFIX}")
    print("=" * 60)
    
    # Procesar documentos (ajustar max_documents según necesidad)
    processed, errors = process_documents_batch(max_documents=500)
    
    # Mostrar resumen
    print("\n" + "=" * 60)
    print("RESUMEN DE PROCESAMIENTO")
    print("=" * 60)
    print(f"Documentos procesados exitosamente: {len(processed)}")
    print(f"Documentos con errores: {len(errors)}")
    
    total_chunks = sum(item["chunks"] for item in processed)
    print(f"Total de chunks creados: {total_chunks}")
    
    if processed:
        print("\nÚltimo documento procesado:")
        last = processed[-1]
        print(f"  ID: {last['document_id']}")
        print(f"  Original: {last['original']}")
        print(f"  Procesado: {last['processed']}")
        print(f"  Chunks: {last['chunks']}")
    
    if errors:
        print("\nÚltimo error:")
        last_error = errors[-1]
        print(f"  Archivo: {last_error['original']}")
        print(f"  Error: {last_error['error'][:200]}...")
    
    return processed, errors

# Ejecutar procesamiento
if __name__ == "__main__":
    processed, errors = main()

PROCESAMIENTO EXTRACTED → PROCESSED
Processed container: bronze
Documentos fuente: servicio_policia/servicio_policia_extracted/
Documentos destino: servicio_policia/servicio_policia_processed/
Buscando documentos en bronze...
Procesando 232 documentos...
------------------------------------------------------------
[1/232] servicio_policia/servicio_policia_extracted/11001-03-06-000-2010-00097-00(2034).json
Procesando documento: servicio_policia/servicio_policia_extracted/11001-03-06-000-2010-00097-00(2034).json
  ✓ Guardado en processed: servicio_policia/servicio_policia_processed/11001-03-06-000-2010-00097-00(2034).json
  ✓ Chunks guardados: 74
  Documento ID: ce5f68b12699
  Chunks creados: 74

[2/232] servicio_policia/servicio_policia_extracted/12. Decreto 2535 de 1993. Normas armas, municiones y explosivos.json
Procesando documento: servicio_policia/servicio_policia_extracted/12. Decreto 2535 de 1993. Normas armas, municiones y explosivos.json
  ✓ Guardado en processed: servicio_poli